In [1]:
print("hi")

hi


In [2]:
# ==========================================
# EXTERNAL BASELINES: GLTR & DetectGPT (Standard Mode + Sliding Window)
# ==========================================
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import GPT2LMHeadModel, GPT2Tokenizer, pipeline
import random
import warnings
import re

warnings.filterwarnings('ignore')

print("="*60)
print("STARTING HEAVY EXTERNAL BASELINES (FULL TEXT & STANDARD DETECTGPT)")
print("="*60)

# 1. إعداد البيانات
SEED = 999
NUM_PERTURBATIONS = 100     # المعيار الطبيعي لعمل DetectGPT
WINDOW_SIZE = 120

df = pd.read_csv('data/processed/processed_articles.csv')

_, df_test = train_test_split(df, test_size=0.20, random_state=SEED, stratify=df['is_AI'])

df_human = df_test[df_test['is_AI'] == 0]
df_ai = df_test[df_test['is_AI'] == 1]

n_samp = min(len(df_human), len(df_ai))

df_sampled = pd.concat([
    df_human.sample(n_samp, random_state=SEED),
    df_ai.sample(n_samp, random_state=SEED)
]).sample(frac=1, random_state=SEED).reset_index(drop=True)

# 2. تحميل النماذج
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading Models on {device}...")

tokenizer_gpt = GPT2Tokenizer.from_pretrained("gpt2-medium")
model_gpt = GPT2LMHeadModel.from_pretrained("gpt2-medium").to(device)
model_gpt.eval()

mask_filler = pipeline(
    "fill-mask", 
    model="distilroberta-base", 
    device=0 if device == "cuda" else -1,
    top_k=1
)

# 3. دوال الاستخراج الأساسية
def get_gltr_features(text):
    inputs = tokenizer_gpt(text, return_tensors="pt", truncation=True, max_length=512).to(device)
    if inputs.input_ids.size(1) < 2: return [0.0, 0.0, 0.0, 0.0]
        
    with torch.no_grad():
        outputs = model_gpt(**inputs)
        logits = outputs.logits[0, :-1, :] 
        labels = inputs.input_ids[0, 1:] 
        
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    ranks = (sorted_indices == labels.unsqueeze(1)).nonzero(as_tuple=True)[1]
    
    total = len(ranks)
    if total == 0: return [0.0, 0.0, 0.0, 0.0]
    
    top_10 = (ranks < 10).sum().item() / total
    top_100 = ((ranks >= 10) & (ranks < 100)).sum().item() / total
    top_1000 = ((ranks >= 100) & (ranks < 1000)).sum().item() / total
    out_1000 = (ranks >= 1000).sum().item() / total
    
    return [top_10, top_100, top_1000, out_1000]

def get_log_likelihood(text):
    inputs = tokenizer_gpt(text, return_tensors="pt", truncation=True, max_length=512).to(device)
    if inputs.input_ids.size(1) < 2: return 0.0
    with torch.no_grad():
        outputs = model_gpt(**inputs, labels=inputs.input_ids)
        return -outputs.loss.item() 

def perturb_text(text, mask_prob=0.15):
    """
    استبدال أجزاء من النص بـ <mask> وجعل نموذج Roberta يتنبأ بها.
    """
    # فحص طول النص الفعلي لتجنب انهيار الذاكرة الرسومية
    # نستخدم مقطع النصوص الخاص بـ mask_filler للتحقق من الطول
    tokenized_len = len(mask_filler.tokenizer.encode(text, add_special_tokens=False))
    if tokenized_len > 500: 
        return text # تجاوز النص الحد المسموح، نرجع النص الأصلي لتجنب الخطأ
        
    words = text.split()
    if len(words) < 10: return text
    
    num_masks = max(1, int(len(words) * mask_prob))
    mask_indices = random.sample(range(len(words)), min(num_masks, 25))
    
    for idx in mask_indices:
        words[idx] = "<mask>"
    
    masked_text = " ".join(words)
    
    try:
        # تمرير النص للنموذج
        results = mask_filler(masked_text)
        if isinstance(results[0], list):
            for res in results:
                masked_text = masked_text.replace("<mask>", res[0]['token_str'], 1)
        else:
            masked_text = masked_text.replace("<mask>", results[0]['token_str'], 1)
        return masked_text
    except Exception as e:
        return text
    
def get_safe_chunks(text, window_size=120):
    """
    دالة لتنظيف النص وتقسيمه إلى نوافذ آمنة تماماً لمعالجة نموذج Roberta
    """
    text = str(text)
    # 1. إزالة الروابط لأنها تتحول لعدد هائل من الرموز بدون فائدة لغوية
    text = re.sub(r'http\S+', '', text) 
    text = re.sub(r'www\.\S+', '', text)
    
    # 2. كسر أي "كلمة" متصلة تتجاوز 80 حرفاً لإجبار المرمز على التعامل معها كأجزاء
    text = re.sub(r'(\S{80})', r'\1 ', text) 
    
    words = text.split()
    chunks = [" ".join(words[i:i+window_size]) for i in range(0, len(words), window_size)]
    
    safe_chunks = []
    for chunk in chunks:
        # 3. الفلترة الصارمة: نتحقق من عدد الرموز الفعلي للمقطع
        token_len = len(mask_filler.tokenizer.encode(chunk, add_special_tokens=False))
        
        # الحد الأقصى هو 512، نستخدم 450 لترك هامش أمان ممتاز لعملية التعديل
        if token_len <= 450: 
            safe_chunks.append(chunk)
            
    return safe_chunks

def process_full_text_sliding_window(text):
    """
    تقسيم النص الطويل إلى نوافذ أصغر لتجنب أخطاء الذاكرة في GPU
    """
    # نستخدم الدالة الجديدة بدلاً من text.split()
    chunks = get_safe_chunks(text, window_size=WINDOW_SIZE)
    
    if not chunks:
        return None
        
    chunk_gltr = []
    chunk_orig_ll = []
    chunk_pert_ll = []
    
    for chunk in chunks:
        if len(chunk.split()) < 10: continue
        
        # استخراج ميزات GLTR للمقطع
        chunk_gltr.append(get_gltr_features(chunk))
        
        # حساب احتمالية النص الأصلي
        orig_ll = get_log_likelihood(chunk)
        chunk_orig_ll.append(orig_ll)
        
        # حساب احتمالية النص بعد التعديل (Perturbed)
        p_lls = []
        for _ in range(NUM_PERTURBATIONS):
            p_text = perturb_text(chunk)
            p_lls.append(get_log_likelihood(p_text))
            
        chunk_pert_ll.append(np.mean(p_lls) if p_lls else orig_ll)
        
    if not chunk_gltr:
        return None
        
    # حساب المتوسطات لجميع المقاطع الخاصة بهذا النص
    avg_gltr = np.mean(chunk_gltr, axis=0).tolist()
    avg_orig_ll = np.mean(chunk_orig_ll)
    avg_pert_ll = np.mean(chunk_pert_ll)
    discrepancy = avg_orig_ll - avg_pert_ll
    
    return avg_gltr, avg_orig_ll, avg_pert_ll, discrepancy

# 5. استخراج الميزات
print("\nExtracting Features with Sliding Window (Processing... )")

gltr_features = []
detectgpt_features = []
valid_true_labels = []

for idx, text in enumerate(tqdm(df_sampled['Text'].astype(str).tolist(), desc="Processing Samples")):
    try:
        result = process_full_text_sliding_window(text)
        if result is None: continue
            
        avg_gltr, avg_orig_ll, avg_pert_ll, discrepancy = result
        
        gltr_features.append(avg_gltr)
        detectgpt_features.append([avg_orig_ll, avg_pert_ll, discrepancy])
        valid_true_labels.append(df_sampled['is_AI'].iloc[idx])
        
    except Exception as e:
        continue

# 6. التقييم وبناء النماذج
print(f"\nSuccessfully processed {len(valid_true_labels)} out of {len(df_sampled)} samples.")
print("Evaluating Baselines...")

X_gltr = np.array(gltr_features)
X_dgpt = np.array(detectgpt_features)
y_true = np.array(valid_true_labels)
    
    for chunk in chunks:
        if len(chunk.split()) < 10: continue
        
        # استخراج ميزات GLTR للمقطع
        chunk_gltr.append(get_gltr_features(chunk))
        

clf_gltr = LogisticRegression(random_state=SEED)
clf_dgpt = LogisticRegression(random_state=SEED)

y_pred_gltr = cross_val_predict(clf_gltr, X_gltr, y_true, cv=5)
y_pred_dgpt = cross_val_predict(clf_dgpt, X_dgpt, y_true, cv=5)

def calculate_metrics(y_t, y_p):
    acc = accuracy_score(y_t, y_p)
    prec, rec, f1, _ = precision_recall_fscore_support(y_t, y_p, average='macro', zero_division=0)
    return acc, prec, rec, f1

gltr_acc, gltr_p, gltr_r, gltr_f1 = calculate_metrics(y_true, y_pred_gltr)
dgpt_acc, dgpt_p, dgpt_r, dgpt_f1 = calculate_metrics(y_true, y_pred_dgpt)

results_data = [
    {'Model': 'GLTR (Approximation)', 'Accuracy': gltr_acc, 'Precision': gltr_p, 'Recall': gltr_r, 'F1-Score': gltr_f1},
    {'Model': 'DetectGPT (Zero-Shot)', 'Accuracy': dgpt_acc, 'Precision': dgpt_p, 'Recall': dgpt_r, 'F1-Score': dgpt_f1}
]

df_results = pd.DataFrame(results_data)
df_results.to_csv('results/tables/T12_Heavy_Baselines_FullText.csv', index=False)

print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)
print(df_results.to_string(index=False))
print("="*60)

STARTING HEAVY EXTERNAL BASELINES (FULL TEXT & STANDARD DETECTGPT)
Loading Models on cuda...


Some weights of the model checkpoint at distilroberta-base were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0



Extracting Features with Sliding Window (Processing... )


Processing Samples:   0%|          | 0/2918 [00:00<?, ?it/s]`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Processing Samples: 100%|██████████| 2918/2918 [19:39:27<00:00, 24.25s/it]    



Successfully processed 2917 out of 2918 samples.
Evaluating Baselines...

FINAL RESULTS
                Model  Accuracy  Precision   Recall  F1-Score
 GLTR (Approximation)  0.851903   0.852255 0.851897  0.851864
DetectGPT (Zero-Shot)  0.857731   0.857855 0.857727  0.857717
